In [ ]:
%pip install mlflow
%pip install --upgrade jinja2
%pip install --upgrade Flask
%pip install setuptools
%pip install xgboost

In [2]:
import mlflow

In [3]:
from mlflow import MlflowClient
from pprint import pprint
from sklearn.ensemble import RandomForestRegressor


#### **Analyse exploratoire de données**

In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Charger les données (remplace par le vrai nom de ton fichier)
df = pd.read_csv('data/Loan_Data.csv')

# Afficher les 5 premières lignes
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['customer_id','default']) # On enlève la cible et l'ID
y = df['default'] # La cible est la colonne 'default'


# Split du dataset (On garde 20% des données pour le test final)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Nombre de clients pour l'entraînement : {len(X_train)}")
print(f"Nombre de clients pour le test : {len(X_test)}")


Nombre de clients pour l'entraînement : 8000
Nombre de clients pour le test : 2000


#### **Création de l'expérience**

In [ ]:
TRACKING_URI = ("http://localhost:8080")
client = MlflowClient(tracking_uri=TRACKING_URI)

In [ ]:

experiment_description = (
   "Évaluation des modèles pour la prédiction du défaut de paiement sur des prêts personnels bancaires."
)


experiment_tags = {# tags servant à l'étiquettage et filtrage de nos expériences
    "project_name": "Pret_Bancaire_MLOps",
    "team_sector": "Banque_Detail",  
    "mlops_stage": "Model_Engineering",
    "mlflow.note.content": "experiment_description"
}


# Get or create the Experiment to avoid RESOURCE_ALREADY_EXISTS
exp_name = "Logistic_regression"
exp = client.get_experiment_by_name(exp_name)

if exp is None:
    experiment_id = client.create_experiment(name=exp_name, tags=experiment_tags)
else:
    experiment_id = exp.experiment_id

print("experiment_id:", experiment_id)